# Mở rộng thí nghiệm cho phản biện: lưới W/λ, MATH500, sampling, SVAMP

Trả lời các mục "nên làm" còn lại trong `comment.txt`:

| Phần | Phản biện | Nội dung | Thời gian (T4) |
|---|---|---|---|
| A. `grid` | B2.2 | Lưới mở rộng W ∈ {4, 8, 16, 32} × λ ∈ {1,1; 1,25; 1,5; 2,0} chạy **online** trên 100 câu dev (GSM8K 0–99), ngân sách 1.024 | ~30–45 phút |
| B. `math500` | B2.1 | MATH500 (500 bài, khó hơn), ngân sách tới 4.096: `standard`, RPDI-EE16, cắt cứng N = 100 | ~2,5–3 giờ |
| C. `gsm8k_500_sampling_s0` | B2.1 | Cùng 500 câu GSM8K (100–599) nhưng **sampling** T = 0,6, top-p = 0,95 (cấu hình DeepSeek khuyến nghị), 16.384 token | ~1,5–2 giờ |
| D. `svamp` (tùy chọn) | B2.1 | SVAMP 1.000 bài, 4.096 token | ~30–45 phút |

W, λ của RPDI-EE16 giữ nguyên (16; 1,5) ở B, C, D: đây là kiểm tra **chuyển giao**, không chỉnh lại.

**Cách chạy:** cell 1–4 → cell 5 (smoke test ~5 phút) → cell 6 (chạy nền cả 4 phần, nối tiếp nhau). Theo dõi bằng cell 7. Phần nào xong thì chạy cell phân tích của phần đó (cell 8–11) ngay, không cần chờ hết.

**Nếu bị ngắt:** chạy lại cell 1–4 rồi cell 6. Mỗi phần tự bỏ qua các lượt đã lưu trên Drive.

**Vì sao sampling vẫn dùng được mẹo "một lượt 16k cho mọi ngân sách":** mỗi câu có seed riêng (`--sampling_seed`), và B token đầu của một mẫu 16k có đúng phân phối của một mẫu sinh với ngân sách B (quyết định phanh chỉ phụ thuộc tiền tố).

## 1. Kiểm tra GPU và môi trường

In [ ]:
import os, subprocess
# Không dựa vào thư mục /kaggle: máy Colab cũng có sẵn thư mục này
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
print("Môi trường:", "Kaggle" if IN_KAGGLE else "Colab")
try:
    gpus = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip()
except FileNotFoundError:  # máy CPU không có nvidia-smi
    gpus = ""
print(gpus or "KHÔNG THẤY GPU")
if not gpus:
    raise RuntimeError("Chưa bật GPU. Colab: Runtime → Change runtime type → T4 GPU. Kaggle: Settings → Accelerator → GPU T4 x2.")
if "P100" in gpus:
    raise RuntimeError("vLLM không chạy trên P100. Kaggle: Settings → Accelerator → GPU T4 x2.")

def vllm_busy():
    """PID các tiến trình run_vllm.py đang chạy nền (T4 không đủ bộ nhớ cho hai tiến trình vLLM)."""
    return subprocess.run(["pgrep", "-f", "vllm_engine/run_vllm.py"], capture_output=True, text=True).stdout.split()

if vllm_busy():
    print("LƯU Ý: đang có run_vllm.py chạy nền, PID", ", ".join(vllm_busy()), "- GPU đang bận.")

## 2. Lấy code mới nhất

In [ ]:
REPO_DIR = "/kaggle/working/slm-reasoning-project" if IN_KAGGLE else "/content/slm-reasoning-project"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/luw-cid/slm-reasoning-project.git {REPO_DIR}
%cd {REPO_DIR}
!git pull
!git log --oneline -1

## 3. Cài vLLM (bản V0, có logits processor theo từng request)

vLLM 0.6.3 chỉ hỗ trợ Python ≤ 3.12, nên cell này tạo một môi trường Python 3.12 riêng ở `/tmp/vllm-env` và cho các lệnh `!python` ở những cell sau dùng môi trường đó. Môi trường có sẵn của Colab/Kaggle không bị đụng tới, nên **không cần khởi động lại runtime**. Nếu kernel bị khởi động lại, chỉ cần chạy lại cell này.

In [ ]:
# vLLM 0.6.3 (engine V0, có logits processor theo từng request) chỉ chạy trên Python <= 3.12,
# trong khi Kaggle/Colab có thể đã lên Python 3.13. Tạo môi trường Python 3.12 riêng bằng uv.
import os, subprocess
VENV = "/tmp/vllm-env"   # ngoài /kaggle/working để không bị lưu vào output của Kaggle
PYBIN = f"{VENV}/bin/python"
CHECK = ("import vllm, torch, transformers; print('vllm', vllm.__version__, '| torch', torch.__version__, "
         "'| transformers', transformers.__version__, '| CUDA', torch.cuda.is_available())")

if not (os.path.exists(PYBIN) and subprocess.run([PYBIN, "-c", "import vllm"], capture_output=True).returncode == 0):
    print("Đang cài môi trường Python 3.12 + vLLM 0.6.3 (khoảng 3–6 phút)...")
    !pip install -q uv
    !python -m uv venv {VENV} --python 3.12
    !python -m uv pip install --python {PYBIN} "vllm==0.6.3.post1" "transformers==4.46.3" pytest 2>&1 | tail -n 15

# Từ đây, mọi lệnh `!python ...` ở các cell sau sẽ dùng Python của môi trường này.
# Nếu kernel bị khởi động lại, chạy lại cell này (lần sau chỉ mất vài giây).
os.environ["PATH"] = f"{VENV}/bin:" + os.environ["PATH"]
check = subprocess.run(["python", "-c", CHECK], capture_output=True, text=True)
print(check.stdout or check.stderr[-3000:])
if check.returncode != 0:
    raise RuntimeError("Cài vLLM thất bại. Gửi toàn bộ output của cell này cho Claude.")
!python -m pytest tests -q

## 4. Thư mục kết quả

## 4. Thư mục kết quả và danh sách công việc

In [ ]:
if IN_KAGGLE:
    OUT_ROOT = "/kaggle/working/vllm_runs"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_ROOT = "/content/drive/MyDrive/slm_rlvr/vllm_runs"

DELTA = 3.0  # biên δ đã chốt trong bài, KHÔNG đổi
R16 = "rpdi_legacy:16:1.5"
R, H = "rpdi_legacy_W16_l1.5", "hardtrunc_N100"  # tên phương pháp trong kết quả
MAIN3 = f"standard,{R16},hardtrunc:100"
GRID = ",".join(f"rpdi_legacy:{W}:{lam}" for W in (4, 8, 16, 32) for lam in (1.1, 1.25, 1.5, 2.0))
JOBS = [  # chạy theo đúng thứ tự này
    dict(key="grid", dataset="gsm8k", start=0, end=100, max_tokens=1024, budgets="512,768,1024",
         methods=f"standard,hardtrunc:128,{GRID}", extra=""),
    dict(key="math500", dataset="math500", start=0, end=500, max_tokens=4096, budgets="1024,1536,2048,3072,4096",
         methods=MAIN3, extra=""),
    dict(key="gsm8k_500_sampling_s0", dataset="gsm8k", start=100, end=600, max_tokens=16384,
         budgets="512,768,1024,1536,2048,3072,4096,8192,16384", methods=MAIN3,
         extra="--temperature 0.6 --top_p 0.95 --sampling_seed 0"),
    dict(key="svamp", dataset="svamp", start=0, end=1000, max_tokens=4096, budgets="512,768,1024,1536,2048,4096",
         methods=MAIN3, extra=""),
]
for j in JOBS:
    j["dir"] = f"{OUT_ROOT}/{j['key']}"
    j["total"] = (j["end"] - j["start"]) * len(j["methods"].split(","))
    os.makedirs(j["dir"], exist_ok=True)
    print(f"{j['key']:<24} {j['total']:>5} lượt -> {j['dir']}")

def job_cmd(j):
    return (f"python -u method_c_rpdi_ee/vllm_engine/run_vllm.py --dataset {j['dataset']} --start {j['start']} "
            f"--end {j['end']} --max_tokens {j['max_tokens']} --budgets {j['budgets']} --methods {j['methods']} "
            f"--output_dir {j['dir']} {j['extra']} >> {j['dir']}/log.txt 2>&1")

def chain_busy():
    return subprocess.run(["pgrep", "-f", "run_ext_all.sh"], capture_output=True, text=True).stdout.split()

## 5. Smoke test: 2 bài MATH500 và 2 câu GSM8K có sampling, ngân sách 1.024

In [ ]:
import json
if vllm_busy() or chain_busy():
    raise RuntimeError("GPU đang bận. Đợi lần chạy đó xong rồi mới chạy smoke test.")
SMOKE = f"{OUT_ROOT}/smoke_ext"
!rm -rf {SMOKE}
!python -u method_c_rpdi_ee/vllm_engine/run_vllm.py --dataset math500 --start 0 --end 2 --max_tokens 1024 --budgets 512,1024 --methods {MAIN3} --output_dir {SMOKE}/math
!python -u method_c_rpdi_ee/vllm_engine/run_vllm.py --dataset gsm8k --start 100 --end 102 --max_tokens 1024 --budgets 512,1024 --methods {MAIN3} --temperature 0.6 --top_p 0.95 --sampling_seed 0 --output_dir {SMOKE}/sampling

m = [json.loads(l) for l in open(f"{SMOKE}/math/budgets.jsonl", encoding="utf-8")]
s = [json.loads(l) for l in open(f"{SMOKE}/sampling/raw.jsonl", encoding="utf-8")]
assert len(m) == 12 and len(s) == 6, (len(m), len(s))
for r in m:
    print(f"MATH id={r['sample_id']} {r['method']:<22} @{r['max_new_tokens']}: dự đoán={r['predicted_answer']!r} "
          f"đáp án={r['ground_truth']!r} đúng={r['is_correct']}")
assert all(r["grader"] == "math" for r in map(json.loads, open(f"{SMOKE}/math/raw.jsonl", encoding="utf-8")))
assert all(r["sampling"]["temperature"] == 0.6 for r in s)
print("\nSMOKE TEST ĐẠT")

## 6. Chạy nền cả 4 phần (nối tiếp nhau)

Bỏ phần D (SVAMP) bằng cách đổi `RUN = JOBS[:3]`. Chạy lại cell này sau khi bị ngắt là được: mỗi phần tự bỏ qua lượt đã xong.

In [ ]:
RUN = JOBS  # hoặc JOBS[:3] để bỏ SVAMP
if vllm_busy() or chain_busy():
    raise RuntimeError("Đang có lần chạy khác. Theo dõi bằng cell 7.")
script = "set +e\n" + "\n".join(job_cmd(j) for j in RUN) + "\n"
open("/tmp/run_ext_all.sh", "w").write(script)
print(script)
os.system("nohup bash /tmp/run_ext_all.sh > /tmp/run_ext_all.out 2>&1 &")

## 7. Theo dõi tiến độ (chạy lại bao nhiêu lần cũng được)

In [ ]:
import re
print("Tiến trình:", "ĐANG CHẠY" if (vllm_busy() or chain_busy()) else "không chạy")
for j in JOBS:
    raw, log = f"{j['dir']}/raw.jsonl", f"{j['dir']}/log.txt"
    n = sum(1 for _ in open(raw, encoding="utf-8")) if os.path.exists(raw) else 0
    text = open(log, encoding="utf-8", errors="ignore").read() if os.path.exists(log) else ""
    state = "ĐÃ XONG" if "Hoàn tất" in text or n >= j["total"] else ("đang chạy" if text and n < j["total"] else "chưa chạy")
    print(f"{j['key']:<24} {n:>5}/{j['total']:<5} {state}")
    beats = re.findall(r"\[đang sinh\] (\d+) lượt chưa xong, dài nhất (\d+) token", text)
    if state == "đang chạy" and beats:
        print(f"{'':<24} đang sinh {beats[-1][0]} lượt, dài nhất {int(beats[-1][1]):,} token")
    errs = [l for l in text.splitlines() if "Error" in l or "Traceback" in l]
    if errs:
        print(f"{'':<24} CÓ LỖI, gửi Claude: !tail -40 {log}")
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader

## 8. Phân tích A: lưới W, λ (100 câu dev)

In [ ]:
import json, collections
J = JOBS[0]
recs = [json.loads(l) for l in open(f"{J['dir']}/budgets.jsonl", encoding="utf-8")]
acc = collections.defaultdict(dict); ex = {}
for b in (768, 1024):
    by = collections.defaultdict(list)
    for r in recs:
        if r["max_new_tokens"] == b:
            by[r["method"]].append(r)
    for m, rs in by.items():
        acc[m][b] = 100 * sum(r["is_correct"] for r in rs) / len(rs)
        if b == 768:
            ex[m] = 100 * sum(r["early_exited"] for r in rs) / len(rs)
lines = ["| W \\ λ | 1.1 | 1.25 | 1.5 | 2.0 |", "|---|---|---|---|---|"]
for b in (768, 1024):
    lines.append(f"| **{b} token** | | | | |")
    for W in (4, 8, 16, 32):
        lines.append(f"| {W} | " + " | ".join(f"{acc[f'rpdi_legacy_W{W}_l{l}'][b]:.0f} ({ex[f'rpdi_legacy_W{W}_l{l}']:.0f}%)"
                                             for l in (1.1, 1.25, 1.5, 2.0)) + " |")
lines.append(f"\nStandard: {acc['standard'][768]:.0f} / {acc['standard'][1024]:.0f}; "
             f"cắt cứng N=128: {acc['hardtrunc_N128'][768]:.0f} / {acc['hardtrunc_N128'][1024]:.0f} (768 / 1.024 token). "
             "Trong ngoặc: tỷ lệ phanh.")
out = "\n".join(lines)
open(f"{J['dir']}/grid.md", "w", encoding="utf-8").write(out)
from IPython.display import Markdown, display
display(Markdown(out))

## 9. Phân tích B: MATH500

In [ ]:
from IPython.display import Markdown, display
J = JOBS[1]
pairs = [f"{R}@{b}:standard@{b}" for b in (1024, 2048, 4096)] + [f"{H}@{b}:{R}@{b}" for b in (1024, 2048, 4096)]
pairs += [f"{R}@2048:standard@4096", f"{H}@2048:standard@4096"]
PAIRS = " ".join(pairs)
!python method_c_rpdi_ee/vllm_engine/analyze_budgets.py --budgets_file {J['dir']}/budgets.jsonl --compare {PAIRS} --delta {DELTA} --output_md {J['dir']}/analysis.md --output_json {J['dir']}/analysis.json > /dev/null
display(Markdown(open(f"{J['dir']}/analysis.md", encoding="utf-8").read()))

## 10. Phân tích C: GSM8K có sampling

In [ ]:
from IPython.display import Markdown, display
J = JOBS[2]
pairs = [f"{R}@{b}:standard@{b}" for b in (768, 1024)] + [f"{H}@{b}:{R}@{b}" for b in (768, 1024)]
pairs += [f"{R}@1024:standard@16384", f"{H}@1024:standard@16384", f"{R}@16384:standard@16384"]
PAIRS = " ".join(pairs)
!python method_c_rpdi_ee/vllm_engine/analyze_budgets.py --budgets_file {J['dir']}/budgets.jsonl --compare {PAIRS} --delta {DELTA} --output_md {J['dir']}/analysis.md --output_json {J['dir']}/analysis.json > /dev/null
display(Markdown(open(f"{J['dir']}/analysis.md", encoding="utf-8").read()))

## 11. Phân tích D: SVAMP

In [ ]:
from IPython.display import Markdown, display
J = JOBS[3]
pairs = [f"{R}@{b}:standard@{b}" for b in (512, 768, 1024)] + [f"{H}@{b}:{R}@{b}" for b in (512, 768, 1024)]
pairs += [f"{R}@1024:standard@4096"]
PAIRS = " ".join(pairs)
!python method_c_rpdi_ee/vllm_engine/analyze_budgets.py --budgets_file {J['dir']}/budgets.jsonl --compare {PAIRS} --delta {DELTA} --output_md {J['dir']}/analysis.md --output_json {J['dir']}/analysis.json > /dev/null
display(Markdown(open(f"{J['dir']}/analysis.md", encoding="utf-8").read()))

## 12. Gửi lại kết quả

Nén các file nhỏ (không kèm `raw.jsonl`) rồi tải `ket_qua_ext.zip` về (Colab: từ Drive `MyDrive/slm_rlvr/vllm_runs/`; Kaggle: panel Output). Giải nén vào `results/` của repo.

In [ ]:
keep = "budgets.jsonl analysis.md analysis.json grid.md run_info.json log.txt".split()
files = [f"{j['key']}/{f}" for j in JOBS for f in keep if os.path.exists(f"{j['dir']}/{f}")]
FILES = " ".join(files)
!cd {OUT_ROOT} && rm -f ket_qua_ext.zip && zip -q ket_qua_ext.zip {FILES} && ls -la ket_qua_ext.zip